<a href="https://colab.research.google.com/github/solasobambo-prog/ts-data-science/blob/main/week3e.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mastery Gate - Module 3

1. A column is 55 percent missing. What are your options, and what decides between them?

• Options: I can drop the column, drop the rows with missing values, impute (fill) the missing values with a statistic or placeholder, or flag and keep it by creating a boolean missingness indicator (col_missing) before filling.

Two things decide: sample size impact and whether missingness carries a signal (NMAR vs. MAR). Dropping rows destroys over half your dataset, so it is almost always wrong at 55% missing. I will test whether the outcome rate differs between missing and non-missing rows, if missingness correlates with the target (like missing income predicting higher churn or default), I will flag and keep it; if it is pure random noise with no predictive value or business meaning (like a sparse referral code), I either convert it to a boolean flag or drop the column.

2. Why fill a skewed revenue column with the median rather than the mean?

• Resistance to Outliers: The mean pulls toward extreme values in the long tail, so filling missing rows with the mean injects an artificially inflated number that misrepresents a typical customer.

• Preserving the Center: The median sits at the exact 50th percentile and ignores how far out the extremes sit, preserving the true center of the majority without distorting the distribution.

3. What exactly do the IQR fences flag, and what rule do they implement?

• The Rule: They implement Tukey's fence rule, setting the lower boundary at \(Q_1 - 1.5 \times \text{IQR}\) and the upper boundary at \(Q_3 + 1.5 \times \text{IQR}\), where \(\text{IQR} = Q_3 - Q_1\).

• What They Flag: They flag statistical outliers; any observation falling strictly below the lower fence or strictly above the upper fence—using percentiles rather than the mean and standard deviation so extreme values cannot mask moderate anomalies.

4. What does errors='coerce' do in pd.to_datetime, and why is it useful?

• What It Does: Instead of crashing your script with a ValueError or ParserError when it encounters unparseable text (like 'not recorded' or 'N/A'), it silently converts those invalid entries into NaT (Not a Time).

• Why It Is Useful: It lets you convert an entire messy date column in one pass without halting your pipeline, while giving you a clean count of broken entries via .isna().sum(), though you should pair it with format='mixed' when dates appear in multiple valid styles so legitimate formats are not turned into NaT.

5. The mean is 182,000 and the median is 96,000. What does that tell you about the distribution?

• Strong Right Skewness: The mean is nearly double (about 90% higher than) the median, which indicates a heavily right-skewed distribution.

• What It Means Practically: Most observations sit clustered around or below 96,000, while a relatively small number of extreme high values pull the average sharply upward, meaning you should report the median and IQR rather than the mean and standard deviation to describe a typical value honestly.


In [5]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(21)
n = 48

cities = rng.choice(['Lagos', 'Abuja', 'Kano', 'Ibadan'], size=n)
styles = [lambda s: s, lambda s: s.upper(), lambda s: s.lower(), lambda s: ' ' + s + ' ']
city = [styles[rng.integers(0, 4)](c) for c in cities]

category = rng.choice(['Electronics', 'Home', 'Fashion'], size=n)
category = [c.lower() if rng.random() < 0.3 else c for c in category]

amount = np.round(rng.lognormal(6.6, 0.45, size=n) / 10) * 10
amount[11] = 52000
amount[33] = 8800
amount_txt = ['₦{:,.0f}'.format(a) if rng.random() < 0.7 else '{:,.0f}'.format(a) for a in amount]

dates = pd.to_datetime('2026-09-01') + pd.to_timedelta(rng.integers(0, 60, size=n), unit='D')
formats = ['%Y-%m-%d', '%b %d, %Y', '%Y/%m/%d']
date_txt = [d.strftime(formats[rng.integers(0, 3)]) for d in dates]
date_txt[5] = 'unknown'
date_txt[29] = 'unknown'

rating = rng.integers(1, 6, size=n).astype(float)
rating[rng.choice(n, 7, replace=False)] = np.nan

coupon = [('SAVE10' if rng.random() < 0.45 else None) for _ in range(n)]

order_id = np.arange(8001, 8001 + n).astype(float)
order_id[3] = np.nan

raw = pd.DataFrame({
    'order_id': order_id,
    'customer': rng.choice(['Ada', 'Femi', 'Zainab', 'Chidi', 'Tunde', 'Amaka'], size=n),
    'city': city,
    'category': category,
    'amount': amount_txt,
    'order_date': date_txt,
    'rating': rating,
    'coupon_code': coupon,
})

near_copy = raw.iloc[[4]].copy()
near_copy['city'] = ' ' + near_copy['city'].str.strip().str.upper() + ' '
raw = pd.concat([raw, raw.iloc[[7, 15]], near_copy], ignore_index=True)

# Keep an untouched copy before changing anything
raw_copy = raw.copy()
df = raw.copy()

print(f"Loaded raw shape: {df.shape}")


Loaded raw shape: (51, 8)


In [6]:
print("=== 1. MISSING VALUE REPORT (FROM RAW_COPY) ===")
missing_report = pd.DataFrame({
    'missing_count': raw_copy.isna().sum(),
    'missing_pct': (raw_copy.isna().mean() * 100).round(1)
}).sort_values(by='missing_count', ascending=False)
display(missing_report)

# 1. DROP: Remove rows missing order_id
# Justification: order_id is the primary transaction key; records without an ID cannot be audited or deduplicated reliably.
df = df.dropna(subset=['order_id']).copy()

# 2. FLAG: Create missingness indicators before filling or dropping
# Justification (rating): Flagging missing ratings preserves any behavioral signal behind non-response before imputing.
# Justification (coupon_code): Assumption to verify with data owner — missing coupon_code likely means no promo was applied. Converting to a single boolean flag captures this cleanly without redundant string fills.
df['rating_missing'] = df['rating'].isna()
df['used_coupon'] = df['coupon_code'].notna()
df = df.drop(columns=['coupon_code'])

# 3. FILL: Impute rating with median
# Justification (rating): Median preserves the central score of discrete 1-5 ratings without introducing non-integer drift.
df['rating'] = df['rating'].fillna(df['rating'].median())

print("\nRemaining missing values after Step 1:", df.isna().sum().sum())

=== 1. MISSING VALUE REPORT (FROM RAW_COPY) ===


,missing_count,missing_pct
coupon_code,26,51.0
rating,7,13.7
order_id,1,2.0
customer,0,0.0
category,0,0.0
city,0,0.0
order_date,0,0.0
amount,0,0.0



Remaining missing values after Step 1: 0


In [7]:
print("=== 2. TYPES & DATES ===")
# Strip Naira symbol (₦) and commas, then coerce to float
df['amount'] = pd.to_numeric(
    df['amount'].str.replace('₦', '', regex=False).str.replace(',', '', regex=False),
    errors='coerce'
)

# Parse mixed date formats safely and flag unrecorded ('unknown') dates
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce', format='mixed')
df['date_missing'] = df['order_date'].isna()

df['order_month'] = df['order_date'].dt.to_period('M')
df['order_day'] = df['order_date'].dt.day_name()

print(f"Amount dtype: {df['amount'].dtype} | Missing amounts: {df['amount'].isna().sum()}")
print(f"Order date dtype: {df['order_date'].dtype} | Missing dates flagged: {df['date_missing'].sum()}")

print("\n=== 3. CATEGORIES (BEFORE VS AFTER) ===")
print(f"City unique count -> Before: {raw_copy['city'].nunique()} | After: {df['city'].str.strip().str.title().nunique()}")
print(f"Category unique count -> Before: {raw_copy['category'].nunique()} | After: {df['category'].str.strip().str.title().nunique()}")

df['city'] = df['city'].str.strip().str.title()
df['category'] = df['category'].str.strip().str.title()

print("\nCleaned City Value Counts:")
print(df['city'].value_counts())
print("\nCleaned Category Value Counts:")
print(df['category'].value_counts())


=== 2. TYPES & DATES ===
Amount dtype: int64 | Missing amounts: 0
Order date dtype: datetime64[ns] | Missing dates flagged: 2

=== 3. CATEGORIES (BEFORE VS AFTER) ===
City unique count -> Before: 16 | After: 4
Category unique count -> Before: 6 | After: 3

Cleaned City Value Counts:
city
Ibadan    16
Abuja     14
Lagos     11
Kano       9
Name: count, dtype: int64

Cleaned Category Value Counts:
category
Home           19
Electronics    19
Fashion        12
Name: count, dtype: int64


In [8]:
print("=== 4. DEDUPLICATION ===")
# Justification: order_id is the unique primary key representing true order identity; deduplicating on it after text normalization catches both exact copies and formatting near-duplicates.
rows_before_dedup = len(df)
df = df.drop_duplicates(subset=['order_id']).copy()

print(f"Rows before dedup: {rows_before_dedup} | Rows after dedup: {len(df)} (Removed {rows_before_dedup - len(df)} duplicate rows)")
print(f"Remaining duplicate order_ids: {df.duplicated(subset=['order_id']).sum()}")


print("\n=== 5. OUTLIERS (IQR RULE ON CLEANED AMOUNT) ===")
q1 = df['amount'].quantile(0.25)
q3 = df['amount'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print(f"Amount Fences -> Lower: {lower_fence:.2f} | Upper: {upper_fence:.2f}")
outliers_df = df[(df['amount'] < lower_fence) | (df['amount'] > upper_fence)]
print(f"Outliers flagged: {len(outliers_df)}")
display(outliers_df[['order_id', 'customer', 'city', 'category', 'amount', 'order_date']])

# Decision: Flag all IQR outliers and cap extreme values at the upper fence in a separate modeling column (amount_capped).
# Justification: Flagging preserves genuine high-value orders for financial reporting, while capping at the upper fence prevents extreme values (like 52,000) from distorting mean-based statistics.
df['is_amount_outlier'] = (df['amount'] < lower_fence) | (df['amount'] > upper_fence)
df['amount_capped'] = df['amount'].clip(upper=upper_fence)

print(f"\nRaw Cleaned Amount -> Mean: {df['amount'].mean():.2f} | Median: {df['amount'].median():.2f}")
print(f"Capped Amount      -> Mean: {df['amount_capped'].mean():.2f} | Median: {df['amount_capped'].median():.2f}")


=== 4. DEDUPLICATION ===
Rows before dedup: 50 | Rows after dedup: 47 (Removed 3 duplicate rows)
Remaining duplicate order_ids: 0

=== 5. OUTLIERS (IQR RULE ON CLEANED AMOUNT) ===
Amount Fences -> Lower: -207.50 | Upper: 1732.50
Outliers flagged: 2


,order_id,customer,city,category,amount,order_date
11,8012.0,Amaka,Ibadan,Electronics,52000,2026-10-01
33,8034.0,Tunde,Lagos,Electronics,8800,2026-09-06



Raw Cleaned Amount -> Mean: 2001.49 | Median: 670.00
Capped Amount      -> Mean: 781.60 | Median: 670.00


### Final Before & After Summary

1. **Rows In:** The pipeline started with **51 rows** in `raw_copy` (48 base orders plus 2 exact duplicates and 1 near-duplicate disguised by messy city whitespace and casing).
2. **Rows Out:** After dropping 1 row missing `order_id` and removing all 3 duplicate records on the primary key, the cleaned dataset retains **47 unique orders**.
3. **Statistic That Moved Most:** The **mean of `amount`** moved most dramatically, dropping by 61% from **2,001.49** down to **781.60** after capping the two Electronics outliers (52,000 and 8,800) at the **1,732.50** upper IQR fence, while the median stayed steady at **670.00**.
4. **Missingness Tracking Note:** Two `'unknown'` order dates became `NaT` during datetime parsing (carrying 6 total missing values across `order_date`, `order_month`, and `order_day`), which are now explicitly tracked via the `date_missing` boolean flag.
